# Schema Build — Relational Model, Data Load and Feature Engineering

Construct the analytical database from the profiled source extract.

The preceding notebook, `01_data_inventory.ipynb`, established what the source data contains and which fields can be trusted. This notebook acts on those findings: it builds a four-table relational schema, loads the extract into it, validates the load, and derives the feature and customer value layers.

**Division of responsibility:** all schema definition, transformation, feature engineering and lifetime value logic is written in SQL and held in the `sql/` directory. Python is used only to open the database connection, fetch the source file, and execute the SQL files. No transformation is performed in Python. This mirrors production practice, where the warehouse performs the work and the analytical layer consumes the result.

**Execution order:** the files are numbered and must be run in sequence. `01_schema.sql` drops and recreates all four tables, so it is safe to re-run at any point; `02_load.sql` will fail if run twice without it, as the primary keys reject duplicate rows.

---

## 1. Create the database and build the schema

Open a DuckDB connection and execute `01_schema.sql`. DuckDB is used because it runs in-process without a server, keeping the project reproducible on any machine while retaining full SQL semantics.

Confirm that all four tables have been created.

In [1]:
import duckdb

con = duckdb.connect("../telco.duckdb")

with open("../sql/01_schema.sql") as f:
    con.execute(f.read())

# Confirm the four tables exist
con.execute("SELECT table_name FROM information_schema.tables ORDER BY 1").df()

,table_name
0,billing
1,customer
2,geography
3,subscription
4,vw_customer_clv
5,vw_customer_features
6,vw_customer_value_segments


**Result:** four tables created — `billing`, `customer`, `geography`, `subscription`.

## 2. Inspect the structure of each table

Print the column definitions of each table to confirm that data types, primary keys and nullability match the design.

Verify in particular that `churn_reason`, `churn_score` and `cltv` are the only fields permitting nulls. Every other field is mandatory, and a null appearing in one would indicate a fault in the load.

In [2]:
con.execute("DESCRIBE customer").df()

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,NO,PRI,None,None
1,zip_code,INTEGER,NO,NaN,None,None
2,gender,VARCHAR,NO,NaN,None,None
3,senior_citizen,VARCHAR,NO,NaN,None,None
4,partner,VARCHAR,NO,NaN,None,None
5,dependents,VARCHAR,NO,NaN,None,None
6,tenure_months,INTEGER,NO,NaN,None,None
7,churn_label,VARCHAR,NO,NaN,None,None
8,churn_value,INTEGER,NO,NaN,None,None
9,churn_reason,VARCHAR,YES,NaN,None,None


In [3]:
con.execute("DESCRIBE geography").df()

,column_name,column_type,null,key,default,extra
0,zip_code,INTEGER,NO,PRI,None,None
1,city,VARCHAR,NO,NaN,None,None
2,latitude,DOUBLE,NO,NaN,None,None
3,longitude,DOUBLE,NO,NaN,None,None


In [4]:
con.execute("DESCRIBE billing").df()

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,NO,PRI,None,None
1,paperless_billing,VARCHAR,NO,NaN,None,None
2,payment_method,VARCHAR,NO,NaN,None,None
3,monthly_charges,"DECIMAL(10,2)",NO,NaN,None,None
4,total_charges,"DECIMAL(10,2)",NO,NaN,None,None


In [5]:
con.execute("DESCRIBE subscription").df()

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,NO,PRI,None,None
1,phone_service,VARCHAR,NO,NaN,None,None
2,multiple_lines,VARCHAR,NO,NaN,None,None
3,internet_service,VARCHAR,NO,NaN,None,None
4,online_security,VARCHAR,NO,NaN,None,None
5,online_backup,VARCHAR,NO,NaN,None,None
6,device_protection,VARCHAR,NO,NaN,None,None
7,tech_support,VARCHAR,NO,NaN,None,None
8,streaming_tv,VARCHAR,NO,NaN,None,None
9,streaming_movies,VARCHAR,NO,NaN,None,None


**Result:** all four tables match the intended definition. Primary keys are registered, currency fields are typed `DECIMAL(10,2)`, and only the three expected fields permit nulls.

## 3. Retrieve the source extract and expose it to the database

Download the source workbook via `kagglehub` and register it with the DuckDB connection under the name `raw`.

Registration makes the spreadsheet queryable as though it were a table, without copying it into the database. Note the boundary being observed here: pandas retrieves the file, and does nothing further. Every rename, type conversion and correction is performed in SQL in the following step.

Confirm the record count before proceeding.

In [6]:
import pandas as pd, kagglehub

path = kagglehub.dataset_download("yeanzc/telco-customer-churn-ibm-dataset")
raw = pd.read_excel(f"{path}/Telco_customer_churn.xlsx")

con.register("raw", raw)

c:\Users\IC Clearwater\OneDrive\Documents\GitHub\Customer_Lifetime_Value_Churn_Prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
con.execute("SELECT COUNT(*) FROM raw").df()

,count_star()
0,7043


**Result:** 7,043 records visible to the database, matching the figure established during profiling.

## 4. Load the four tables

Execute `02_load.sql`, which populates each table in dependency order — `geography` first, as `customer` holds a foreign key referring to it.

Two transformations occur during the load, both deliberate:

- **`geography` is deduplicated.** The extract holds 7,043 rows but only 1,652 postal codes, each repeating once per customer resident there. `SELECT DISTINCT` reduces this to one row per postal code. Profiling confirmed that each postal code resolves to exactly one city, latitude and longitude, so no conflicting rows can arise.
- **`total_charges` is converted from text to numeric.** The eleven blank entries identified during profiling are resolved to `0.00` rather than discarded. All eleven are newly acquired customers with zero tenure who have not yet been billed; zero is the accurate value, not a missing one. Retaining these records preserves the new-customer population, which carries the highest churn exposure in the dataset.

Confirm the row count of each table after loading.

In [8]:
with open("../sql/02_load.sql") as f:
    con.execute(f.read())

con.execute("SELECT COUNT(*) FROM geography").df()

,count_star()
0,1652


In [9]:
for f in ["../sql/01_schema.sql", "../sql/02_load.sql"]:
    con.execute(open(f).read())

con.execute("SELECT COUNT(*) FROM customer").df()

,count_star()
0,7043


In [10]:
for f in ["../sql/01_schema.sql", "../sql/02_load.sql"]:
    con.execute(open(f).read())

con.execute("SELECT COUNT(*) FROM subscription").df()

,count_star()
0,7043


In [11]:
for f in ["../sql/01_schema.sql", "../sql/02_load.sql"]:
    con.execute(open(f).read())

con.execute("SELECT COUNT(*) FROM billing").df()

,count_star()
0,7043


In [12]:
for f in ["../sql/01_schema.sql", "../sql/02_load.sql"]:
    con.execute(open(f).read())

con.execute("""
    SELECT 'geography' AS t, COUNT(*) FROM geography
    UNION ALL SELECT 'customer', COUNT(*) FROM customer
    UNION ALL SELECT 'subscription', COUNT(*) FROM subscription
    UNION ALL SELECT 'billing', COUNT(*) FROM billing
""").df()

,t,count_star()
0,geography,1652
1,customer,7043
2,subscription,7043
3,billing,7043


**Result:** 1,652 rows in `geography` and 7,043 in each of `customer`, `subscription` and `billing`, as intended.

## 5. Validate the load

Row counts alone do not establish that the data loaded correctly. Run a set of checks against known values established during profiling, so that any silent corruption is detected before the data is built upon.

The checks are held in `sql/02b_validate.sql` rather than written inline, so that the validation logic remains visible alongside the rest of the SQL layer. Each check returns a figure whose expected value is already known from the profiling stage; a check returning anything else indicates a fault in the load rather than a finding in the data.

In [13]:
con.execute(open("../sql/02b_validate.sql").read()).df()

,check,value
0,churned customers,1869
1,churn_reason populated,1869
2,zero total_charges,11
3,zero tenure,11
4,full four-table join,7043


The final check is the most significant. A join returning fewer than 7,043 rows would indicate that customers had been lost when the flat extract was separated into four tables. No records were lost.

## 6. Build the feature engineering layer

Execute `03_features.sql`, which creates `vw_customer_features`. The layer is implemented as a view rather than a materialised table: at 7,043 rows the performance difference is immaterial, and a view keeps the derivation of every feature readable rather than concealed inside stored data.

Seven features are derived: tenure banding, internet and telephone flags, a count of optional services held, a contract risk ranking, an automatic payment indicator, and a monthly spend band.

Examine the churn rate within each tenure band to confirm the band boundaries are meaningful.

In [14]:
con.execute(open("../sql/03_features.sql").read())
con.execute("SELECT tenure_band, COUNT(*), AVG(churn_value) FROM vw_customer_features GROUP BY 1 ORDER BY 1").df()

,tenure_band,count_star(),avg(churn_value)
0,Established,1856,0.255388
1,Loyal,3001,0.119294
2,New,2186,0.474382


Nearly half of first-year customers depart, falling to approximately one in eight beyond three years. Churn risk is concentrated overwhelmingly in the first year of the relationship — a finding that shapes where retention effort should be directed irrespective of any model output.

## 7. Examine the add-on count, and the encoding issue it exposes

Count the optional services each customer holds and examine churn within each level.

This is the feature most exposed to the encoding problem identified during profiling. The six add-on fields hold three values, not two: `Yes`, `No`, and `No internet service`. The third value does not indicate a declined sale — it indicates that the service was never available, because the customer holds no internet subscription at all.

In [15]:
con.execute(open("../sql/03_features.sql").read())
con.execute("SELECT addon_count, COUNT(*), AVG(churn_value) FROM vw_customer_features GROUP BY 1 ORDER BY 1").df()

,addon_count,count_star(),avg(churn_value)
0,0,2219,0.214060
1,1,966,0.457557
2,2,1033,0.358180
3,3,1118,0.273703
4,4,852,0.223005
5,5,571,0.124343
6,6,284,0.052817


**Result:** the pattern is consistent from one add-on upward — each additional service held reduces churn — but the zero add-on group breaks it, returning an unexpectedly low 21.4%.

The cause is that the zero group contains two entirely different populations. Separate them using the `has_internet` flag.

In [16]:
con.execute("""
    SELECT has_internet, addon_count, COUNT(*) AS customers, AVG(churn_value) AS churn_rate
    FROM vw_customer_features
    GROUP BY 1, 2 ORDER BY 1, 2
""").df()

,has_internet,addon_count,customers,churn_rate
0,0,0,1526,0.074050
1,1,0,693,0.522367
2,1,1,966,0.457557
3,1,2,1033,0.358180
4,1,3,1118,0.273703
5,1,4,852,0.223005
6,1,5,571,0.124343
7,1,6,284,0.052817


The lowest-risk customers in the dataset and the highest-risk customers in the dataset were occupying the same category, their averages concealing both. Encoding `No internet service` as `No` — the conventional default in most preparation routines — would have merged them permanently and silently.

Once separated, the internet-subscribing population forms an uninterrupted progression: 52.2%, 45.8%, 35.8%, 27.4%, 22.3%, 12.4%, 5.3%. Each additional service a customer holds makes their departure materially less likely.

## 8. Derive customer lifetime value

Execute `04_clv.sql`, which creates `vw_customer_clv`.

The `CLTV` field supplied with the dataset was rejected during profiling: it is bounded between 2,003 and 6,500 while total charges reach 8,684, and it shows negligible correlation with its own determinants. Lifetime value is therefore derived independently here, and is deliberately not validated against the supplied field.

**The calculation rests on three assumptions, none of which are present in the data. They are stated in full in `04_clv.sql` and repeated here:**

1. **Gross margin of 30%.** Revenue is not profit, and the costs of serving a customer are absent from this dataset. Thirty percent falls within the normal range for telecommunications operators.
2. **Expected remaining tenure assigned by contract type** — twelve months for month-to-month, twenty-four for one year, thirty-six for two year. The dataset is a single point-in-time snapshot with no dates, so true expected tenure cannot be calculated; survival analysis requires a time dimension that does not exist here. Contract type is used as a proxy because it is the strongest available indicator of commitment.
3. **No discounting applied.** Over horizons of thirty-six months or less, at these amounts, discounting does not alter any intervention decision. The omission is deliberate.

Examine the resulting value distribution by contract type.

In [17]:
con.execute(open("../sql/04_clv.sql").read())
con.execute("""
    SELECT contract, COUNT(*) AS customers,
           ROUND(MIN(clv),2) AS min_clv,
           ROUND(AVG(clv),2) AS avg_clv,
           ROUND(MAX(clv),2) AS max_clv
    FROM vw_customer_clv GROUP BY 1 ORDER BY 3
""").df()

,contract,customers,min_clv,avg_clv,max_clv
0,Month-to-month,3875,67.50,239.03,422.82
1,One year,1473,131.40,468.35,853.92
2,Two year,1695,198.72,656.32,1282.50


The customers most likely to depart are also the least valuable, while the most valuable customers are contractually committed and rarely leave. This establishes the central argument of the project: a retention list ranked by churn probability alone directs expenditure toward the least valuable segment of the customer base.

## 9. Segment customers by value

Apply `NTILE(10)` to divide the customer base into ten equally sized groups ranked by lifetime value, where decile 1 holds the highest-value customers.

Deciles are used in preference to absolute amounts because intervention decisions are taken on relative standing — which customers rank highest — rather than on specific currency thresholds.

In [18]:
con.execute(open("../sql/04_clv.sql").read())
con.execute("""
    SELECT value_decile,
           COUNT(*) AS customers,
           ROUND(AVG(clv),2) AS avg_clv,
           ROUND(AVG(churn_value),3) AS churn_rate,
           ROUND(SUM(clv),2) AS total_value
    FROM vw_customer_value_segments
    GROUP BY 1 ORDER BY 1
""").df()

,value_decile,customers,avg_clv,churn_rate,total_value
0,1,705,1044.42,0.055,736317.18
1,2,705,723.70,0.138,510208.56
2,3,705,486.96,0.125,343307.16
3,4,704,354.67,0.517,249687.36
4,5,704,311.12,0.487,219027.96
5,6,704,276.15,0.411,194406.30
6,7,704,241.55,0.259,170054.46
7,8,704,199.72,0.188,140600.52
8,9,704,153.37,0.233,107973.54
9,10,704,80.99,0.243,57018.60


Multiplying each decile's total value by its churn rate gives the value genuinely at risk:

- **Deciles 4 to 6: R316k at risk** — 54% of all value exposed, from 30% of customers
- Deciles 1 to 3: R154k at risk
- Deciles 7 to 10: R109k at risk

The highest-value customers are not at risk, and the customers at greatest risk are not the most valuable. The retention opportunity lies in the middle deciles, which a model ranked on churn probability alone would not identify.

---

## Summary

1. A four-table relational schema was constructed from the flat source extract, with primary and foreign key constraints enforced.
2. All 7,043 records were loaded, with the eleven unbilled accounts retained and corrected rather than discarded.
3. The load was validated against five known values, including a full four-table join confirming no records were lost in the separation.
4. Seven features were derived in SQL, including the internet flag that separates customers who declined an optional service from those to whom it was never offered — a distinction worth a sevenfold difference in churn rate.
5. Customer lifetime value was derived independently of the rejected supplied field, on three explicitly stated assumptions.
6. Value segmentation identified deciles 4 to 6 as holding the majority of value at risk.

**Next stage:** model development in `03_modelling.ipynb` — logistic regression and gradient boosting, with logistic regression carried forward on grounds of regulatory explainability.